In [83]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float64

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [85]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [86]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [87]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
R = matrix_mf.R
R

tensor([[ 0.,  0., 62.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0., 84.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [44., 60.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ..., 92.,  0.,  0.]], device='mps:0')

In [88]:
alpha = matrix_mf.compute_alpha().item()
R *= alpha
alpha

0.10527777671813965

In [89]:
num_latent_factors = 3
lmf = LogisticMatrixFactorization(
    R=R,
    num_factors=num_latent_factors,
    alpha=alpha,
    lambd=0.01,
    device=device,
    dtype=dtype,
)

num_epochs = 10000
lmf.train_with_gradients(
    num_epochs=num_epochs,
    learning_rate=0.01,
    log_interval=10,
)

Epoch 1: loss = 3446.721923828125, MPR = 0.4922444522380829
Epoch 11: loss = 3135.263671875, MPR = 0.4756205976009369
Epoch 21: loss = 3024.877685546875, MPR = 0.4630703032016754
Epoch 31: loss = 2936.736328125, MPR = 0.45201921463012695
Epoch 41: loss = 2863.133544921875, MPR = 0.44179868698120117
Epoch 51: loss = 2800.279541015625, MPR = 0.43237194418907166
Epoch 61: loss = 2745.841552734375, MPR = 0.42365503311157227
Epoch 71: loss = 2698.212158203125, MPR = 0.41512277722358704
Epoch 81: loss = 2656.20654296875, MPR = 0.4071062207221985
Epoch 91: loss = 2618.91455078125, MPR = 0.39938291907310486
Epoch 101: loss = 2585.61669921875, MPR = 0.39205092191696167
Epoch 111: loss = 2555.731201171875, MPR = 0.38506555557250977
Epoch 121: loss = 2528.78173828125, MPR = 0.37828996777534485
Epoch 131: loss = 2504.37255859375, MPR = 0.3720034062862396
Epoch 141: loss = 2482.17138671875, MPR = 0.3662884533405304
Epoch 151: loss = 2461.897216796875, MPR = 0.3605768084526062
Epoch 161: loss = 2443

In [90]:
lmf.save("models", "lmf_all_types_100")
lmf = LogisticMatrixFactorization.load(os.path.join("models", "lmf_all_types_100.pt"))

In [91]:
px.line(x=range(len(lmf.losses)), y=lmf.losses.cpu(), title="Loss").show()
px.line(x=range(len(lmf.mprs)), y=lmf.mprs.cpu(), title="MPRS").show()

In [92]:
user_id = matrix_mf.usernames_to_ids(["michelle"])[0]

# Get the top 10 recommendations for the user
top_10_ids, top_10_ids_scores = lmf.recommend(user_id, top_k=20, filter_user_items=True)

# Get the top 10 recommendations for the user
matrix_mf.items_ids_to_df(top_10_ids)[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
1,jaslkh,ROSALÍA,SAOKO,2022,70,0.827,0.768,0.2650,0.7900,0.000024,0.4970,0.7340,99.988,-5.702,137533,2022,70
46,jaslkh,Khruangbin,So We Won't Forget,2020,65,0.641,0.413,0.1020,0.0305,0.844000,0.1830,0.9670,101.270,-13.212,298333,2020,65
61,jaslkh,Etta James,At Last,1960,79,0.274,0.348,0.0293,0.5470,0.013300,0.3340,0.3280,87.430,-8.631,179693,1960,79
103,jaslkh,Sebastian Yatra,Dos Oruguitas,2021,69,0.423,0.355,0.0454,0.7610,0.000000,0.0915,0.4650,93.788,-10.565,214613,2021,69
105,jaslkh,Feyesal,please never fall in love again,2019,53,0.713,0.345,0.0374,0.3640,0.002520,0.3930,0.3470,107.637,-8.550,157642,2019,53
110,jaslkh,Poupie,Feux (feat. Jul),2020,44,0.858,0.645,0.3710,0.4780,0.000000,0.0707,0.4860,132.085,-5.564,197296,2020,44
111,jaslkh,Khruangbin,So We Won't Forget,2020,65,0.641,0.413,0.1020,0.0305,0.844000,0.1830,0.9670,101.270,-13.212,298333,2020,65
113,jaslkh,Joji,Glimpse of Us,2022,84,0.440,0.317,0.0531,0.8910,0.000005,0.1410,0.2680,169.914,-9.258,233456,2022,84
115,jaslkh,Luidji,Gisèle - Part 4 - Piano Session,2019,45,0.607,0.316,0.0688,0.8100,0.000030,0.1230,0.3050,86.890,-13.653,299873,2019,45
133,jaslkh,Ashh,Motel,2015,67,0.393,0.627,0.0537,0.2170,0.010200,0.1030,0.0966,92.937,-10.743,241352,2015,67


In [93]:
if lmf.num_factors <= 3:
    df_tracks = df_matrix_mf.copy()
    df_tracks = df_tracks.sample(frac=1) # Shuffle the dataframe
    df_tracks = df_tracks[:1000] # Keep only 1000 tracks

    # Add item latent factors to the dataframe
    items_latent_columns = [f"track_latent_{i}" for i in range(lmf.num_factors)]
    for track_id in df_tracks["id"].unique():
        latent_factors = lmf.get_item_latent_factors(matrix_mf.items_to_ids([track_id])[0]).tolist()
        df_tracks.loc[df_tracks["id"] == track_id, items_latent_columns] = latent_factors

    # Add user latent factors to the dataframe
    users_latent_columns = [f"latent_factor_{i}" for i in range(lmf.num_factors)]
    for user_id in df_tracks["username"].unique():
        latent_factors = lmf.get_user_latent_factors(matrix_mf.usernames_to_ids([user_id])[0]).tolist()
        df_tracks.loc[df_tracks["username"] == user_id, users_latent_columns] = latent_factors

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=items_latent_columns,
    ).show()

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=users_latent_columns,
    ).show()